# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [13]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [7]:
y = spaceship["Transported"].astype(int) 
X = spaceship.drop(columns=["Transported","PassengerId","Name"])

In [24]:
X["Cabin"] = X["Cabin"].fillna("Unknown/Unknown/Unknown")

X[["Cabin_Deck", "Cabin_Num", "Cabin_Side"]] = (
    X["Cabin"].str.split("/", expand=True)
)

In [25]:
X = X.drop(columns=["Cabin"])

In [26]:
X = X.drop(columns=["Cabin_Num"])

In [27]:
num_cols = X.select_dtypes(include=["float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

In [28]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [29]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [30]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [34]:
X_transformed = preprocess.fit_transform(X)
X_transformed.shape

(8693, 28)

**Perform Train Test Split**

In [35]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [39]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, classification_report

bagging_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        bootstrap=True,
        random_state=42
    ))
])

bagging_pipe.fit(X_train, y_train)
y_pred = bagging_pipe.predict(X_test)

print("Bagging F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Bagging F1: 0.7868080094228505
              precision    recall  f1-score   support

           0       0.77      0.82      0.80       863
           1       0.81      0.76      0.79       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



- Random Forests

In [40]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_pipe.fit(X_train, y_train)
y_pred = rf_pipe.predict(X_test)

print("Random Forest F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Random Forest F1: 0.7938021454112039
              precision    recall  f1-score   support

           0       0.78      0.84      0.81       863
           1       0.83      0.76      0.79       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Gradient Boosting

In [41]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=42
    ))
])

gb_pipe.fit(X_train, y_train)
y_pred = gb_pipe.predict(X_test)

print("Gradient Boosting F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Gradient Boosting F1: 0.8175750834260289
              precision    recall  f1-score   support

           0       0.83      0.78      0.80       863
           1       0.80      0.84      0.82       876

    accuracy                           0.81      1739
   macro avg       0.81      0.81      0.81      1739
weighted avg       0.81      0.81      0.81      1739



- Adaptive Boosting

In [42]:
from sklearn.ensemble import AdaBoostClassifier

ada_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    ))
])

ada_pipe.fit(X_train, y_train)
y_pred = ada_pipe.predict(X_test)

print("AdaBoost F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

AdaBoost F1: 0.7463054187192119
              precision    recall  f1-score   support

           0       0.73      0.84      0.78       863
           1       0.81      0.69      0.75       876

    accuracy                           0.76      1739
   macro avg       0.77      0.76      0.76      1739
weighted avg       0.77      0.76      0.76      1739



Which model is the best and why?

Among the ensemble methods tested, Gradient Boosting achieved the best performance, with the highest F1-score and accuracy on the test set.
Compared to Bagging and Random Forests, Gradient Boosting provided a better balance between precision and recall for both classes.
This is likely because boosting methods iteratively focus on correcting previous misclassifications, making them well-suited for capturing complex, non-linear relationships in the data.
AdaBoost showed weaker performance, possibly due to its sensitivity to noise and outliers in the dataset.